##Step 1: Install Dependencies and Dataset

In [ ]:
!pip install opendatasets pandas scikit-learn

In [ ]:
import opendatasets as od

od.download("https://www.kaggle.com/datasets/meetnaren/goodreads-best-books")

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: saimaghonigadepally
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/meetnaren/goodreads-best-books


100%|██████████| 2.13G/2.13G [00:30<00:00, 75.7MB/s]


#Step 2: Load Dataset

In [ ]:
import os

os.listdir("/content")

['.config', 'goodreads-best-books', 'sample_data']

In [ ]:
os.listdir("/content/goodreads-best-books")

['images', 'book_data.csv']

In [ ]:
import pandas as pd
df = pd.read_csv("/content/goodreads-best-books/book_data.csv", on_bad_lines="skip")

In [ ]:
df.columns

Index(['book_authors', 'book_desc', 'book_edition', 'book_format', 'book_isbn',
       'book_pages', 'book_rating', 'book_rating_count', 'book_review_count',
       'book_title', 'genres', 'image_url'],
      dtype='object')

#Step 3: Select Required Columns

In [ ]:
df = df[[
    "book_title",
    "book_authors",
    "book_desc",
    "genres",
    "book_rating",
    "book_rating_count"
]]

#Step 4: Rename Columns

In [ ]:
df.rename(columns={
    "book_title": "title",
    "book_authors": "author",
    "book_desc": "summary"
}, inplace=True)

##Step 5: Clean and Prepare Data

In [ ]:
df = df.sample(min(5000, len(df)), random_state=42).reset_index(drop=True)

# Fill missing
df["title"] = df["title"].fillna("")
df["author"] = df["author"].fillna("")
df["summary"] = df["summary"].fillna("")
df["genres"] = df["genres"].fillna("")

# Create content (ONLY ONCE)
df["content"] = df["title"] + " " + df["author"] + " " + df["summary"] + " " + df["genres"]

# Remove empty rows
df = df[df["content"].str.strip() != ""].reset_index(drop=True)

##Step 6: Mood-Based

In [ ]:
def mood_recommend(mood):
    mood = mood.lower()

    if mood == "happy":
        filtered = df[df["genres"].str.contains("romance|comedy|fun", case=False, na=False)]

    elif mood == "sad":
        filtered = df[df["genres"].str.contains("drama|tragedy|emotional", case=False, na=False)]

    elif mood == "adventure":
        filtered = df[df["genres"].str.contains("adventure|fantasy|action", case=False, na=False)]

    elif mood == "thriller":
        filtered = df[df["genres"].str.contains("thriller|mystery|crime", case=False, na=False)]

    else:
        return ["No mood matched"]

    return filtered["title"].head(5).tolist()

In [ ]:
mood_recommend("happy")
#mood_recommend("adventure")

['Butterfly Tattoo',
 'Until You',
 'Genuine Lies',
 'Warm Bodies',
 'Welcome, Reluctant Stranger']

#Step 7: Content Based

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english")
matrix = tfidf.fit_transform(df["content"])

similarity = cosine_similarity(matrix)

#**Recommendation**


In [ ]:
def recommend(book_name):
    book_name = book_name.lower()

    matches = df[df["title"].str.lower().str.contains(book_name)]

    if matches.empty:
        return ["Book not found"]

    index = matches.index[0]

    distances = list(enumerate(similarity[index]))
    books = sorted(distances, key=lambda x: x[1], reverse=True)[1:6]

    return [df.iloc[i[0]].title for i in books]

##Step 8: Popularity-Based

In [ ]:
import numpy as np

df["book_rating"] = pd.to_numeric(df["book_rating"], errors="coerce").fillna(0)
df["book_rating_count"] = pd.to_numeric(df["book_rating_count"], errors="coerce").fillna(0)

df["popularity_score"] = df["book_rating"] * np.log1p(df["book_rating_count"])

popular_books = df.sort_values(by="popularity_score", ascending=False)

top_popular = popular_books["title"].head(10).tolist()
top_popular

['The Hunger Games',
 'Harry Potter and the Deathly Hallows',
 'Harry Potter en de Relieken van de Dood',
 'Harry Potter and the Deathly Hallows',
 'Harry Potter and the Prisoner of Azkaban',
 'To Kill a Mockingbird',
 'The Hobbit',
 'El hobbit',
 'The Hobbit',
 'Pride and Prejudice']

##Step 9: Saving Pickle Files

In [ ]:
import pickle

pickle.dump(df, open("books.pkl", "wb"))
pickle.dump(similarity, open("similarity.pkl", "wb"))
pickle.dump(popular_books, open("popular.pkl", "wb"))

##Step 10: Test

In [ ]:
print(recommend("harry potter"))
print(mood_recommend("happy"))
print(popular_books[:5])

['De waarheid over de zaak Harry Quebert', 'Het achtste leven (voor Brilka)', 'Harry Potter and the Deathly Hallows', 'Van de koele meren des doods', 'De gelukkige huisvrouw']
['Butterfly Tattoo', 'Until You', 'Genuine Lies', 'Warm Bodies', 'Welcome, Reluctant Stranger']
                                         title                        author  \
657                           The Hunger Games               Suzanne Collins   
1276      Harry Potter and the Deathly Hallows                  J.K. Rowling   
487    Harry Potter en de Relieken van de Dood  J.K. Rowling|Wiebe Buddingh'   
2853      Harry Potter and the Deathly Hallows                  J.K. Rowling   
2855  Harry Potter and the Prisoner of Azkaban    J.K. Rowling|Mary GrandPré   

                                                summary  \
657   Winning will make you famous. Losing means cer...   
1276  Harry Potter is preparing to leave the Dursley...   
487   Harry krijgt de zware taak om de resterende Gr...   
2853  Harry

##Step 11: Download

In [ ]:
from google.colab import files

files.download("books.pkl")
files.download("similarity.pkl")
files.download("popular.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>